# 03 分段插值与三次样条

全局多项式插值用一个多项式穿过所有节点。分段插值换一种思路：把区间切成许多小区间，只在每个小区间上构造低次多项式，然后再考虑相邻小区间之间应该满足什么连接条件。

本节的主线是：

1. 分段线性插值：只要求函数值连续；
2. 分段三次 Hermite 插值：同时匹配函数值和节点导数，使整体达到一阶连续；
3. 自然三次样条插值：不预先给导数，而是通过二阶光滑性和边界条件解出整条曲线。

这三种方法都具有局部性，但光滑性、误差阶和对数据的敏感程度不同。


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
    "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd()
while not (ROOT / "src" / "py_sc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import (
    NaturalCubicSpline,
    lagrange_interpolate,
    piecewise_cubic_hermite_interpolate,
    piecewise_linear_interpolate,
)


## 分段线性插值

设节点按从小到大排列为

$$
x_0 < x_1 < \cdots < x_n, \qquad y_i=f(x_i).
$$

在第 $i$ 个小区间 $[x_i,x_{i+1}]$ 上，记

$$
h_i=x_{i+1}-x_i, \qquad t=\frac{x-x_i}{h_i}, \qquad 0\le t\le 1.
$$

分段线性插值就是在每个小区间上使用一次多项式

$$
L_i(x)=(1-t)y_i+t y_{i+1}.
$$

写回 $x$ 的形式为

$$
L_i(x)=
\frac{x_{i+1}-x}{x_{i+1}-x_i}y_i
+\frac{x-x_i}{x_{i+1}-x_i}y_{i+1}.
$$

这里的两个系数可以看成局部线性基函数。第一个基函数在 $x_i$ 处为 1，在 $x_{i+1}$ 处为 0；第二个基函数在 $x_i$ 处为 0，在 $x_{i+1}$ 处为 1。因此 $L_i(x_i)=y_i$，$L_i(x_{i+1})=y_{i+1}$。

分段线性插值的整体函数是连续的，因为相邻区间在公共节点处给出同一个函数值。但是它的导数在每个小区间内是常数：

$$
L_i'(x)=\frac{y_{i+1}-y_i}{h_i}.
$$

在内部节点 $x_i$ 附近，左侧斜率通常是 $(y_i-y_{i-1})/h_{i-1}$，右侧斜率通常是 $(y_{i+1}-y_i)/h_i$，二者一般不相等。所以分段线性插值通常只有 $C^0$ 连续，而不是 $C^1$ 连续。

如果真实函数 $f\in C^2[x_i,x_{i+1}]$，则在单个小区间上有误差公式

$$
f(x)-L_i(x)=\frac{f''(\xi)}{2}(x-x_i)(x-x_{i+1}),\qquad \xi\in(x_i,x_{i+1}).
$$

于是

$$
|f(x)-L_i(x)|\le \frac{h_i^2}{8}\max_{\xi\in[x_i,x_{i+1}]}|f''(\xi)|.
$$

这说明分段线性插值的误差主要受两个因素控制：网格宽度 $h_i$ 和函数弯曲程度 $|f''|$。网格越细，误差通常越小；函数越弯，直线段越难逼近。


In [ ]:
x = np.array([0.0, 0.8, 1.7, 2.5, 3.2, 4.0])
y = np.cos(1.5 * x) + 0.2 * x
x_eval = np.linspace(x.min(), x.max(), 500)

linear = piecewise_linear_interpolate(x, y, x_eval)
polynomial = lagrange_interpolate(x, y, x_eval)

plt.figure(figsize=(8, 4.5))
plt.plot(x_eval, polynomial, label="全局多项式插值")
plt.plot(x_eval, linear, label="分段线性插值")
plt.scatter(x, y, color="black", zorder=3, label="数据点")
plt.xlabel("x")
plt.ylabel("函数值")
plt.title("全局插值与局部分段插值的对比")
plt.legend()
plt.tight_layout()
plt.show()


## 局部性实验

分段方法的一个重要优点是局部性。对全局多项式插值，修改一个节点值会改变整个插值多项式；对分段线性插值，修改 $y_i$ 只会直接影响 $[x_{i-1},x_i]$ 和 $[x_i,x_{i+1}]$ 两个相邻区间。

这种局部性是分段方法在工程计算中常用的原因之一。它牺牲了高阶光滑性，但换来了更稳定、更容易控制的局部行为。


In [ ]:
y_perturbed = y.copy()
y_perturbed[3] += 0.8

poly_perturbed = lagrange_interpolate(x, y_perturbed, x_eval)
linear_perturbed = piecewise_linear_interpolate(x, y_perturbed, x_eval)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
axes[0].plot(x_eval, polynomial, label="原始数据")
axes[0].plot(x_eval, poly_perturbed, label="扰动后")
axes[0].scatter(x, y_perturbed, color="black", s=20)
axes[0].set_title("全局多项式")
axes[0].set_xlabel("x")
axes[0].set_ylabel("函数值")
axes[0].legend()

axes[1].plot(x_eval, linear, label="原始数据")
axes[1].plot(x_eval, linear_perturbed, label="扰动后")
axes[1].scatter(x, y_perturbed, color="black", s=20)
axes[1].set_title("分段线性")
axes[1].set_xlabel("x")
axes[1].legend()

fig.suptitle("局部方法限制了单个数据变化的影响范围")
fig.tight_layout()
plt.show()


## 分段三次 Hermite 插值

分段线性插值只使用节点函数值。如果还希望曲线在节点处具有给定的切线方向，可以在每个节点同时使用函数值和导数值：

$$
y_i=f(x_i),\qquad m_i\approx f'(x_i).
$$

在区间 $[x_i,x_{i+1}]$ 上仍令

$$
h_i=x_{i+1}-x_i,\qquad t=\frac{x-x_i}{h_i}.
$$

分段三次 Hermite 插值在这个区间上写成

$$
H_i(x)=h_{00}(t)y_i+h_{10}(t)h_i m_i+h_{01}(t)y_{i+1}+h_{11}(t)h_i m_{i+1},
$$

其中四个三次 Hermite 基函数为

$$
h_{00}(t)=2t^3-3t^2+1,
$$

$$
h_{10}(t)=t^3-2t^2+t,
$$

$$
h_{01}(t)=-2t^3+3t^2,
$$

$$
h_{11}(t)=t^3-t^2.
$$

这些基函数的设计目标不是任意的，而是要同时满足端点函数值和端点导数条件：

$$
H_i(x_i)=y_i,\qquad H_i(x_{i+1})=y_{i+1},
$$

$$
H_i'(x_i)=m_i,\qquad H_i'(x_{i+1})=m_{i+1}.
$$

公式中出现 $h_i m_i$ 和 $h_i m_{i+1}$，是因为基函数先对无量纲变量 $t$ 求导，而真实导数满足

$$
\frac{d}{dx}=\frac{1}{h_i}\frac{d}{dt}.
$$

如果同一个节点导数 $m_i$ 同时用于左右两个相邻区间，那么拼接后的函数不仅函数值连续，而且一阶导数也连续。因此分段三次 Hermite 插值通常是 $C^1$ 的。不过它没有要求二阶导数连续，所以一般不是 $C^2$ 的。

如果 $f\in C^4[x_i,x_{i+1}]$，并且端点导数 $m_i=f'(x_i)$、$m_{i+1}=f'(x_{i+1})$ 是准确的，则单区间误差为

$$
f(x)-H_i(x)=\frac{f^{(4)}(\xi)}{4!}(x-x_i)^2(x-x_{i+1})^2.
$$

这比线性插值的误差阶更高。但实际计算中，导数往往不是已知量，需要从数据估计。斜率选得不好时，三次 Hermite 插值可能产生过冲；PCHIP 的核心思想就是用更谨慎的斜率选择规则来保持单调性和形状。


In [ ]:
# 这里用 numpy.gradient 给出一个简单的节点斜率估计。
# 它适合教学演示，但不具备 PCHIP 那样的保单调性质。
slopes = np.gradient(y, x)
hermite = piecewise_cubic_hermite_interpolate(x, y, slopes, x_eval)

plt.figure(figsize=(8, 4.5))
plt.plot(x_eval, linear, label="分段线性插值")
plt.plot(x_eval, hermite, label="分段三次 Hermite 插值")
plt.scatter(x, y, color="black", zorder=3, label="数据点")
plt.quiver(
    x,
    y,
    0.18 * np.ones_like(x),
    0.18 * slopes,
    angles="xy",
    scale_units="xy",
    scale=1,
    color="tab:red",
    width=0.004,
    label="节点斜率方向",
)
plt.xlabel("x")
plt.ylabel("函数值")
plt.title("分段三次 Hermite 插值通过节点斜率控制曲线形状")
plt.legend()
plt.tight_layout()
plt.show()


## 自然三次样条插值

分段三次 Hermite 插值需要先给出每个节点的导数。三次样条插值换一个角度：仍然在每个小区间上使用三次多项式，但不预先指定所有节点导数，而是要求相邻三次多项式在节点处足够光滑。

在区间 $[x_i,x_{i+1}]$ 上设

$$
S_i(x)=a_i+b_i(x-x_i)+c_i(x-x_i)^2+d_i(x-x_i)^3.
$$

三次样条要满足三类条件。

第一，插值条件：

$$
S_i(x_i)=y_i,\qquad S_i(x_{i+1})=y_{i+1}.
$$

第二，内部节点处的一阶导数连续：

$$
S_{i-1}'(x_i)=S_i'(x_i),\qquad i=1,2,\ldots,n-1.
$$

第三，内部节点处的二阶导数连续：

$$
S_{i-1}''(x_i)=S_i''(x_i),\qquad i=1,2,\ldots,n-1.
$$

这些条件使样条函数达到 $C^2$ 连续。和分段三次 Hermite 相比，样条不仅切线连续，曲率也连续，因此视觉上和数值上都更平滑。

从未知量个数看，$n$ 个小区间共有 $4n$ 个三次多项式系数。插值条件给出 $2n$ 个方程，一阶和二阶连续性各给出 $n-1$ 个方程，总共是 $4n-2$ 个方程，还差两个条件。因此必须补充两个边界条件。

自然三次样条（natural cubic spline）选择端点二阶导数为零：

$$
S''(x_0)=0,\qquad S''(x_n)=0.
$$

这可以理解为端点附近不额外施加弯曲。它不是唯一合理的边界条件，只是在没有端点导数信息时常用、简单且稳定。

更常见的推导会引入节点二阶导数

$$
M_i=S''(x_i).
$$

对内部节点 $x_i$，由一阶导数连续性可以得到三对角方程

$$
h_{i-1}M_{i-1}+2(h_{i-1}+h_i)M_i+h_iM_{i+1}
=6\left(\frac{y_{i+1}-y_i}{h_i}-\frac{y_i-y_{i-1}}{h_{i-1}}\right),
$$

其中 $h_i=x_{i+1}-x_i$。自然边界条件给出

$$
M_0=0,\qquad M_n=0.
$$

因此三次样条的核心计算不是直接求一个高次多项式，而是求解一个三对角线性系统。这个结构很重要：它让样条方法既有全局光滑约束，又保持了接近局部的计算成本。


In [ ]:
spline = NaturalCubicSpline.fit(x, y)
spline_values = spline(x_eval)

plt.figure(figsize=(8, 4.5))
plt.plot(x_eval, linear, label="分段线性插值")
plt.plot(x_eval, hermite, label="分段三次 Hermite 插值")
plt.plot(x_eval, spline_values, label="自然三次样条")
plt.scatter(x, y, color="black", zorder=3, label="数据点")
plt.xlabel("x")
plt.ylabel("函数值")
plt.title("分段线性、三次 Hermite 与自然三次样条")
plt.legend()
plt.tight_layout()
plt.show()

print("自然三次样条的各区间三次多项式系数")
for i, (a, b, c, d) in enumerate(zip(spline.a, spline.b, spline.c, spline.d)):
    print(f"区间 {i}: a={a: .4f}, b={b: .4f}, c={c: .4f}, d={d: .4f}")


## 边界条件与方法比较

三次样条的端点条件会影响靠近边界的曲线形状。常见选择包括：

* 自然边界条件：给定 $S''(x_0)=S''(x_n)=0$；
* 固定导数边界条件：给定 $S'(x_0)$ 和 $S'(x_n)$，也称 clamped spline；
* not-a-knot 条件：要求靠近端点的两个三次多项式在更高阶意义上拼接得更自然。

分段三次 Hermite 和三次样条都使用分段三次多项式，但重点不同：

* Hermite 方法把节点导数当作输入，导数选得好时形状可控，通常达到 $C^1$；
* 样条方法通过整体光滑性解出曲线，通常达到 $C^2$，但端点行为受边界条件影响；
* PCHIP 属于分段三次 Hermite 路线，它重点解决普通 Hermite 斜率估计可能带来的过冲问题。

因此，选择方法时应先判断问题更看重什么：如果只需要稳健和局部，分段线性足够；如果知道或能可靠估计导数，可以用分段三次 Hermite；如果希望整体曲率更平滑，可以用三次样条。


## 小结

* 分段线性插值使用局部一次多项式，具有 $C^0$ 连续性，误差主要由网格宽度和二阶导数控制。
* 分段三次 Hermite 插值同时使用函数值和节点导数，通常具有 $C^1$ 连续性；若端点导数准确，单区间误差可达到四阶形式。
* 三次样条插值通过一阶、二阶连续性把相邻三次多项式连接起来，通常具有 $C^2$ 连续性。
* 自然三次样条会导出三对角线性系统，这为后续线性方程组章节提供了自然入口。
* Hermite、PCHIP 和样条都属于分段三次方法，但区别在于导数或曲率信息是如何确定的。
